In [1]:
import os, sys
sys.path.append(os.path.abspath(os.path.join(os.curdir, '..')))
from src.utils import *
from src.iiwa_program import Iiwa14IKProgram
from pydrake.all import (
    StartMeshcat,
    Quaternion,
    RigidTransform,
    MinimumDistanceLowerBoundConstraint,
    SceneGraphInspector,
)

ikflow/config.py | Using device: 'cuda:0'


In [2]:
meshcat = StartMeshcat()
diagram = BuildEnv(meshcat=meshcat, directives_file = os.path.join(RepoDir(), "models/iiwa14/iiwa14_collision.yaml"))

program = Iiwa14IKProgram(diagram)
target_pose = np.array([0.29252776760806476, 0.4714304570247682, 0.8799253594708412, 0.37517094428362757, -0.6461691081943264, 0.40286211881371536, -0.5285965942054528], dtype=np.float32)
program.create_prog(target_pose)

INFO:drake:Meshcat listening for connections at http://localhost:7001


WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
URDFParser: Link size: 11
URDFParser: Joint size: 11
URDFParser: Done loading robot file /home/tangles/.cache/jrl/urdfs/iiwa14_formatted_link_filepaths_absolute.urdf
Initialized robot collision data structures in time 0.227031


In [3]:
# q = np.random.uniform(-np.pi, np.pi, 7)
q = [1.66184657,  0.95890669,  0.95386061,  0.89356231,  0.8742057, -1.04379779, 1.49698538]
program.plant.SetPositions(program.plant_context, q)
print(program.collision_free_constraint.eval_func(q = q))
program.diagram.ForcedPublish(program.diagram_context)
pose = program.frame.CalcPoseInWorld(program.plant_context)
print("[", *np.array([*pose.translation(), *pose.rotation().ToQuaternion().wxyz()]), "]", sep=", ")
print(q)


[0.41599509]
[, 0.29252776760806476, 0.4714304570247682, 0.8799253594708412, 0.37517094428362757, -0.6461691081943264, 0.40286211881371536, -0.5285965942054528, ]
[1.66184657, 0.95890669, 0.95386061, 0.89356231, 0.8742057, -1.04379779, 1.49698538]


In [6]:
program.create_prog(np.array([*pose.translation(), *pose.rotation().ToQuaternion().wxyz()]))

In [7]:
pose = program.target_pose
pose = RigidTransform(Quaternion(pose[3], pose[4], pose[5], pose[6]), [pose[0], pose[1], pose[2]])
DrawAxes(pose, meshcat)

result = program.Solve()

print(result.is_success())

vars = result.get_x_val()

True
